# Module 8: Image Embedding Generation & Vector Search
## Visual Embedding Indexing, Vector Search & Demographic Filtering

This notebook demonstrates:
1. Loading serialized 1,280-dimensional visual feature vectors and metadata index.
2. Sub-millisecond Cosine Similarity nearest-neighbor search across indexed apparel.
3. Applying granular categorical, outfit-part, and demographic filters during retrieval.
4. Visual inspection of top-5 nearest neighbors for query fashion items.

In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import time
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import seaborn as sns

from src.embeddings import EmbeddingManager
from src.config import EMBEDDINGS_NPY, EMBEDDINGS_INDEX_CSV

sns.set_theme(style="white", palette="muted")
plt.rcParams["figure.figsize"] = (14, 7)

### 1. Load Precomputed Visual Embedding Index

In [ ]:
manager = EmbeddingManager()
print(f"Embeddings Loaded : {len(manager):,} garments")
print(f"Matrix Shape      : {manager.embeddings.shape}")
print(f"Index DataFrame   : {manager.index_df.shape}")
print("\nSample indexed records:")
manager.index_df[["id", "canonical_category", "outfit_part", "gender", "productDisplayName"]].head(5)

### 2. Category & Demographic Coverage Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Canonical Category counts
cat_counts = manager.index_df["canonical_category"].value_counts()
sns.barplot(x=cat_counts.values, y=cat_counts.index, ax=axes[0], palette="viridis")
axes[0].set_title("Garments per Canonical Category", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Count")

# Outfit Part counts
part_counts = manager.index_df["outfit_part"].value_counts()
sns.barplot(x=part_counts.values, y=part_counts.index, ax=axes[1], palette="mako")
axes[1].set_title("Garments per Outfit Part", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Count")

plt.tight_layout()
plt.show()

### 3. Sub-Millisecond Nearest-Neighbor Visual Search

We select a query item (e.g. a casual Shirt or dress) and retrieve the top-5 visually closest garments using normalized vector dot-product.

In [ ]:
# Pick a query item from index
query_row = manager.index_df.iloc[10]
query_id = query_row["id"]
query_name = query_row.get("productDisplayName", "Garment")
query_cat = query_row.get("canonical_category", "")

t0 = time.perf_counter()
matches = manager.search_by_item_id(query_id, top_k=5, exclude_self=True)
latency_ms = (time.perf_counter() - t0) * 1000.0

print(f"Query Item ID: {query_id} ({query_cat}) - '{query_name}'")
print(f"Search Latency across {len(manager):,} items: {latency_ms:.3f} ms\n")
for r, m in enumerate(matches, 1):
    print(f"Rank #{r}: ID {m['id']} | Sim: {m['similarity_score']:.4f} | {m['canonical_category']} | {m['productDisplayName']}")

### 4. Visualizing Query Item vs. Retrieved Top Neighbors

In [ ]:
fig, axes = plt.subplots(1, len(matches) + 1, figsize=(18, 4))

# Display Query Image
q_img = Image.open(query_row["image_path"]).convert("RGB")
axes[0].imshow(q_img)
axes[0].set_title(f"QUERY ITEM\nID: {query_id}\n({query_cat})", fontsize=10, color="darkred", fontweight="bold")
axes[0].axis("off")

# Display Top Matches
for i, match in enumerate(matches):
    m_img = Image.open(match["image_path"]).convert("RGB")
    axes[i + 1].imshow(m_img)
    axes[i + 1].set_title(
        f"Match #{i+1} (Sim: {match['similarity_score']:.3f})\nID: {match['id']}\n({match['canonical_category']})",
        fontsize=9,
        color="navy",
    )
    axes[i + 1].axis("off")

plt.suptitle(f"Top-5 Visual Similarity Search for '{query_name}'", fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

### 5. Cross-Part Filtering: Finding Matching Footwear or Bottoms for a Top

In an outfit recommendation system, we often want to find complementary pieces (e.g. given a Top, find the most stylistically aligned Bottoms or Shoes).

In [ ]:
# Filter to find bottoms matching the top's visual style
bottom_matches = manager.search_by_item_id(
    query_id,
    top_k=4,
    outfit_part="bottom",
)

print(f"Found {len(bottom_matches)} matching bottom items for Query {query_id}:")
for r, b in enumerate(bottom_matches, 1):
    print(f"  Option #{r}: ID {b['id']} ({b['canonical_category']}) - Sim: {b['similarity_score']:.4f} - {b['productDisplayName']}")